In [2]:
import torch
import os
import gc
from diffusers import StableDiffusionXLPipeline
import yaml

with open(os.path.expanduser("~/prog/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

In [ ]:
MODEL_PATH = config["paths"]["base_model_dir"]
OUTPUT_DIR = config["paths"]["random_images_dir"]

PROMPT = input("Enter the prompt for image generation (default: 'A photo of a cat'): ") or "A photo of a cat"
NUM_IMAGES = 1

INFERENCE_STEPS = 20

In [ ]:
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)

### If you have at least 16GB VRAM

In [ ]:
pipe.to("cuda")  # Move the pipeline to GPU for faster inference

### Otherwise

In [ ]:
# Enable optimizations
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.enable_xformers_memory_efficient_attention()

### Generation images

In [ ]:
for i in range(NUM_IMAGES):
    image_num = i+1

    current_seed = torch.randint(0, 1000000, (1,)).item()
    generator = torch.Generator("cuda").manual_seed(current_seed)

    filename = f"{image_num:03d}_{current_seed}.png"
    #filename = f"cat_{image_num:03d}_{current_seed}.png"
    filepath = os.path.join(OUTPUT_DIR, filename)

    print(f"Generation {image_num}/{NUM_IMAGES} with seed {current_seed}...")

    try:
        image = pipe(
            prompt=PROMPT,
            num_inference_steps=INFERENCE_STEPS,
            generator=generator,
        ).images[0]

        image.save(filepath)

        del image
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Error generating image {image_num}: {e}")
        continue